# SOXL Smart Leverage Wealth Control

## 假设与边界

复用 `trainable_regime_policy_nested_wf_v1` 在 inner validation 中确定的 outer 目标仓位，只把它视为机会信号。本实验丢弃来源卡中的旧收益列，使用 qlib SOXL/SOXX 开盘价重新执行 `close(t) signal -> open(t+1) trade -> open(t+2) exit`。

论文启发部分是路径依赖的财富锁盈：仅在策略财富跑赢 70% SOXX + 零收益现金基准后降风险，落后时不追损；同时比较纯 SOXL 与保持等效 beta 的 SOXL/SOXX 梯形。2023–2026 outer test 只评估，不选择参数。当前 qlib store 没有 PIT T-bill/T-bond，因此现金收益固定为零，这不是论文的忠实复刻。

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import Markdown, display
from qlib.workflow import R

HERE = Path.cwd().resolve()
ROOT = next(path for path in (HERE, *HERE.parents) if (path / 'src').is_dir() and (path / 'SPEC.md').exists())
if ROOT.as_posix() not in sys.path:
    sys.path.insert(0, ROOT.as_posix())

from src.workflow.smart_leverage_soxl_wealth_control import (
    run_smart_leverage_soxl_wealth_control_card,
)

CARD = ROOT / 'research' / 'smart_leverage_soxl_wealth_control_v1'
summary = run_smart_leverage_soxl_wealth_control_card(root=ROOT)
display({
    'recorder_id': summary['recorder_id'],
    'as_of': summary['data_audit']['as_of'],
    'promotion_decision': summary['promotion_decision'],
    'required_cost_pass_count': summary['required_cost_pass_count'],
})

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


[38797:MainThread](2026-07-21 15:33:11,706) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[38797:MainThread](2026-07-21 15:33:11,707) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[38797:MainThread](2026-07-21 15:33:11,707) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/soxl')}


/Users/rziiiii/dev/workspace/qlib_venv/lib/python3.11/site-packages/mlflow/tracking/_tracking_service/utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
[38797:MainThread](2026-07-21 15:33:12,107) INFO - qlib.workflow - [exp.py:258] - Experiment 335844174620613605 starts running ...


[38797:MainThread](2026-07-21 15:33:12,210) INFO - qlib.workflow - [recorder.py:345] - Recorder 9b3a66708a464e629ee6b066385ab22e starts running under Experiment 335844174620613605 ...


[38797:MainThread](2026-07-21 15:33:12,317) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | waiting `async_log` Done


{'recorder_id': '9b3a66708a464e629ee6b066385ab22e',
 'as_of': '2026-07-20',
 'promotion_decision': 'research_reject_smart_leverage_controller_failed_required_gates',
 'required_cost_pass_count': 0}

## 10bp 策略比较与 outer 年度结果

In [2]:
comparison = pd.read_csv(CARD / summary['sidecars']['strategy_comparison'])
display(comparison.loc[comparison['cost_bps'].eq(10), [
    'strategy', 'return_pct', 'max_drawdown_pct', 'worst_day_pct',
    'cvar5_pct', 'worst_rolling_5d_pct', 'avg_soxl_position_pct',
    'upside_capture_pct', 'calmar', 'matched_beta_calmar',
    'outer_years_risk_passed', 'all_required_gates_pass', 'failed_gates',
]])
annual = pd.read_csv(CARD / summary['sidecars']['annual_outer_metrics'])
display(annual.loc[(annual['cost_bps'].eq(10)) & annual['strategy'].eq(summary['primary_candidate']['strategy'])])

,strategy,return_pct,max_drawdown_pct,worst_day_pct,cvar5_pct,worst_rolling_5d_pct,avg_soxl_position_pct,upside_capture_pct,calmar,matched_beta_calmar,outer_years_risk_passed,all_required_gates_pass,failed_gates
4,nested_opportunity_clean_soxl,627.482027,-62.280280,-15.651596,-7.187345,-27.008219,50.353894,47.712997,1.243458,1.442106,0,False,"risk_gate,upside_capture_gate,calmar_same_posi..."
5,nested_opportunity_beta_ladder,588.484092,-62.332724,-15.487196,-7.258478,-26.897587,28.748846,47.568992,1.197444,1.442106,0,False,"risk_gate,upside_capture_gate,calmar_matched_b..."
6,smart_leverage_asymmetric_lock_soxl,422.295048,-51.795826,-14.059100,-5.333981,-23.924586,38.152397,36.066487,1.182323,1.398548,1,False,"risk_gate,upside_capture_gate,calmar_same_posi..."
7,smart_leverage_asymmetric_lock_beta_ladder,419.574256,-52.496494,-14.192898,-5.475230,-24.311397,14.223172,36.479645,1.161910,1.399681,1,False,"risk_gate,participation_gate,upside_capture_ga..."


,strategy,cost_bps,outer_year,return_pct,cagr_pct,annualized_vol_pct,max_drawdown_pct,worst_day_pct,cvar5_pct,worst_rolling_5d_pct,...,avg_equity_position_pct,avg_effective_beta,upside_capture_pct,calmar,total_turnover,qlib_annualized_return_pct,qlib_information_ratio,qlib_max_drawdown_pct,rows,risk_gate_pass
28,smart_leverage_asymmetric_lock_beta_ladder,10,2023,81.425803,82.732548,42.708743,-27.843070,-7.122218,-5.074455,-13.210442,...,97.477567,1.450808,48.361908,2.971387,51.096279,82.732548,1.423253,-27.843070,249,False
29,smart_leverage_asymmetric_lock_beta_ladder,10,2024,16.833719,16.906160,32.059624,-23.118831,-6.798955,-4.776008,-11.091961,...,83.669057,1.010593,32.558247,0.731272,56.140979,16.906160,0.485645,-23.118831,251,False
30,smart_leverage_asymmetric_lock_beta_ladder,10,2025,24.118239,24.441763,42.533289,-44.773951,-14.192898,-6.744196,-24.311397,...,85.925791,1.132064,34.642884,0.545892,66.846056,24.441763,0.508531,-44.773951,249,False
31,smart_leverage_asymmetric_lock_beta_ladder,10,2026,97.489842,303.186833,41.197374,-12.146255,-7.134614,-4.438506,-9.131295,...,78.776964,0.933637,30.111772,24.961342,25.347448,303.186833,3.414816,-12.146255,123,True


## Recorder 证据与最终结论

In [3]:
recorder = R.get_recorder(
    recorder_id=summary['recorder_id'],
    experiment_name=summary['experiment'],
)
display({
    'recorder_local_dir': recorder.get_local_dir(),
    'recorder_only_artifacts': summary['artifact_policy']['recorder_only_artifacts'],
    'selection_uses_outer_test': summary['selection_uses_outer_test'],
    'bond_data_status': summary['bond_data_status'],
})
display(Markdown((CARD / 'conclusion.md').read_text(encoding='utf-8')))

{'recorder_local_dir': '/Users/rziiiii/dev/workspace/research/smart_leverage_soxl_wealth_control_v1/qlib_recorder/335844174620613605/9b3a66708a464e629ee6b066385ab22e',
 'recorder_only_artifacts': ['artifacts/smart_leverage_soxl_wealth_control_v1/positions.csv',
  'artifacts/smart_leverage_soxl_wealth_control_v1/strategy_returns.csv'],
 'selection_uses_outer_test': False,
 'bond_data_status': 'needs_data_pit_tbill_tbond_not_in_current_qlib_store'}

# 结论

## 跑了什么

复用 `trainable_regime_policy_nested_wf_v1` 的 nested outer 目标仓位，但丢弃旧 `strategy_return`，重新使用 qlib SOXL/SOXX 开盘价按 close(t) 信号、next open 到 following open 收益计算。固定测试非对称财富锁盈和 beta-preserving SOXL/SOXX 梯形配置；outer 2023–2026 不参与控制器参数选择。

## 核心结果

10bp 主方案 `smart_leverage_asymmetric_lock_beta_ladder`：总收益 419.57%，CAGR 61.00%，MaxDD -52.50%，单日最差 -14.19%，CVaR5 -5.48%，滚动五日最差 -24.31%，SOXL 平均仓位 14.22%，上涨捕获 36.48%。正式全门槛通过数为 0。

## 增量效果

相对干净重算的 `nested_opportunity_clean_soxl`，主方案收益变化 -207.91 个百分点，MaxDD 改善 +9.78 个百分点，单日最差改善 +1.46 个百分点，CVaR5 改善 +1.71 个百分点，上涨捕获变化 -11.23 个百分点。

## 决定

`research_reject_smart_leverage_controller_failed_required_gates`。这是论文启发的诊断实现，不是原论文忠实复刻；当前 qlib store 缺少 PIT T-bill/T-bond，债券状态为 `needs_data_pit_tbill_tbond_not_in_current_qlib_store`。此外该架构是在看过既有 outer 结果后提出，任何看似通过的结果仍需方案冻结后的未来 paper-sim 确认。

## 风险与下一步

SOXX 当前成分股特征仍有 survivorship 风险；本卡只复用来源模型的目标仓位，不把旧盘中回放收益作为证据。下一步只在结果显示尾部改善且参与度未坍塌时，才值得补 PIT 债券并训练路径级 LFNN；否则停止该方向。

不构成买卖建议。
